# Kaggriculture: Replay Warehouse и анализ поражений

Тетрадь скачивает все завершённые реплеи выбранной отправки, но при повторном запуске загружает только новые. Она строит компактные датасеты по эпизодам, дням и рыночным решениям, классифицирует стратегии соперников и сохраняет отчёт на Google Drive.

Полные реплеи занимают примерно 30 MB каждый. Для 140 игр потребуется около 4–5 GB свободного места на Drive. Запрос доступа к Drive происходит в первой ячейке.

In [ ]:
# Единственное место с параметрами. None означает последнюю завершённую отправку.
SUBMISSION_ID = 55680902
TEAM_NAME = "Grigorii IU"
MAX_REPLAYS = 0  # 0 = все; для короткой проверки можно поставить 10
FORCE_ANALYSIS = False

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive/Kaggriculture")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Google Drive подключён: {DRIVE_ROOT}")

In [ ]:
# Получаем код и проверяем Kaggle до скачивания гигабайтов данных.
import os
import subprocess
import sys
from google.colab import userdata

REPOSITORY = "https://github.com/GrigoriiIurev/Kaggriculture.git"
PROJECT = Path("/content/Kaggriculture")
if (PROJECT / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=True)
try:
    token = userdata.get("KAGGLE_API_TOKEN")
except Exception:
    token = None
if token:
    os.environ["KAGGLE_API_TOKEN"] = token
check = subprocess.run(["kaggle", "competitions", "submissions", "kaggriculture", "--format", "json"], text=True, capture_output=True)
if check.returncode != 0:
    raise RuntimeError("Kaggle не авторизован. Добавьте KAGGLE_API_TOKEN в Colab Secrets.\n" + check.stderr)
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Kaggle авторизован, версия кода: {commit}")

In [ ]:
command = [
    sys.executable, "-u", "run_replay_analysis_pipeline.py",
    "--drive-root", str(DRIVE_ROOT),
    "--max-replays", str(MAX_REPLAYS),
]
if SUBMISSION_ID is not None:
    command.extend(["--submission-id", str(SUBMISSION_ID)])
if TEAM_NAME:
    command.extend(["--team", TEAM_NAME])
if FORCE_ANALYSIS:
    command.append("--force-analysis")
LOG_PATH = Path("/content/kaggriculture_replay_analysis.log")
print("Запускаю Replay Warehouse:", " ".join(command), flush=True)
with LOG_PATH.open("a", encoding="utf-8") as log:
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)

In [ ]:
# Показываем готовый отчёт и размеры файлов.
import json
from IPython.display import Markdown, display

receipt = json.loads((DRIVE_ROOT / "replay_warehouse/latest_analysis.json").read_text())
ANALYSIS = Path(receipt["analysis"])
display(Markdown((ANALYSIS / "report.md").read_text()))
print("\nФайлы анализа:")
for path in sorted(ANALYSIS.iterdir()):
    print(f"- {path.name}: {path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"\nСырые реплеи: {receipt['sync']['local_selected']}")